![](img/logo_ucm.jpg)

# MLflow

## Introducción a MLflow

MLflow es una plataforma de código abierto para gestionar el ciclo de vida de proyectos de Machine Learning (ML), desde la experimentación hasta la implementación. Es Open Source, por lo que es mantenido por la comunidad aunque hay empresas que aportan y desarrollan en paralelo y es también compatible con cualquier plataforma que soporte Python y Java, además de API REST. Su enfoque modular y escalable permite a los desarrolladores y equipos de ciencia de datos trabajar con mayor eficiencia y reproducibilidad. 

Implementar modelos de ML en producción suele ser un proceso complejo. En muchas organizaciones, la transición desde un prototipo en Jupyter Notebook hasta un sistema robusto y escalable implica fricción entre equipos y tecnologías. Los científicos de datos y los ingenieros de ML suelen trabajar con herramientas, estándares y objetivos distintos. Mientras los primeros se centran en experimentar con datos y modelos, los segundos deben garantizar que esos modelos se integren adecuadamente en aplicaciones de software, bajo condiciones exigentes de rendimiento y fiabilidad.

Un escenario común es que, tras entrenar un modelo, los ingenieros deban reimplementarlo para poder servirlo (teniendo que realizar procesos de reingenería). Esto introduce varios desafíos:

- Errores al reescribir modelos creados por otros equipos.
- Problemas de escalabilidad al servir predicciones.
- Dificultades para reproducir entrenamientos por falta de entornos estandarizados.

MLflow aborda directamente estos problemas al proporcionar herramientas que permiten a cualquier profesional del ML gestionar experimentos, guardar modelos, registrar versiones y desplegarlos en entornos de producción de forma controlada y repetible.

`````{admonition} Databricks
:class: note
MLflow fue inicialmente creado en Databricks y lanzado como código abierto como plataforma para ayudar en la implementación de sistemas de ML. MLflow permite a un profesional cotidiano en una plataforma gestionar el ciclo de vida de ML, desde la iteración en el desarrollo del modelo hasta la implementación en un entorno confiable y escalable que es compatible con los requisitos de los sistemas de software [modernos](https://mlflow.org/docs/latest/index.html).
`````


```{figure} img/mlflow/mlflow_components.png
---
name: mlflow components
width: 50%
---
Componentes del ecosistema MLflow.
```

## Primer modelo

Partiremos del modelo de ejemplo que hemos venido utilizando durante el curso. 

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [4]:
dataset = datasets.load_iris()

X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=99)

clf = LogisticRegression()
clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.9333333333333333

Pero esto es solo una pequeña parte del proceso de Ingeniería de ML. Como se puede intuir, es necesario crear código adicional para poner en producción este modelo y asegurarse de que el código de entrenamiento anterior sea reutilizable y repetible. Uno de los principales objetivos de MLflow es ayudar en el proceso de configuración de sistemas y proyectos de ML. 

Con unas pocas líneas de código más, podemos iniciar la primera interacción con MLflow. 

In [5]:
import mlflow

mlflow.sklearn.autolog()

mlflow.set_experiment("my_experiment_1")

with mlflow.start_run():
    clf = LogisticRegression(random_state=99)
    clf.fit(X_train, y_train)
    clf.score(X_test, y_test)


La instrucción **mlflow.sklearn.autolog()** permite registrar automáticamente el experimento en el directorio local. Por defecto, los metadatos de una ejecución de MLflow se almacenan en el sistema de archivos local. En este directorio, se almacena la información relevante para realizar un seguimiento de los **experimentos**. **MLflow Tracking** es el módulo responsable de manejar estos registros. 

Cuando usas el keyword **with**, Python busca en el objeto los métodos especiales:

    __enter__(): lo que se ejecuta al entrar al bloque.

    __exit__(): lo que se ejecuta al salir del bloque, incluso si hubo excepciones.

Ejemplo clásico, leer un fichero, queremos que Python maneje el I/O a fichero.
```python
with open("archivo.txt", "r") as f:
    contenido = f.read()
```

En nuestro caso, mlflow.start_run() devuelve un context manager que se encargará de iniciar el experimento y cerrarlo. Tras el lanzamiento, los resultados se encuentran en la ruta *mlruns*.



```{figure} img/mlflow/mlruns_0.png
---
name: primera ejecución
width: 50%
---
Primera ejecución de experimento.
```

`````{admonition} mlruns
:class: important
El sistema de ficheros **mlruns** de MLflow almacena todos los metadatos, métricas, parámetros y artefactos de cada ejecución de experimentos de ML. Esto permite rastrear, comparar y reproducir *experimentos* de manera organizada.
`````

## Contenido registrado por autolog, por defecto: 

### 1. Metadatos (`meta.yaml`)

- `0/meta.yaml`: Información general del experimento (`experiment_id = 0`).
- `0/<run_id>/meta.yaml`: Información de la ejecución específica (fecha, estado, duración, etc.).

También se incluyen:

- `inputs/.../meta.yaml`: Datos de entrada registrados.
- `datasets/.../meta.yaml`: Información del dataset usado.


### 2. Métricas (`metrics/`)

El experimento ha registrado varias métricas de rendimiento del modelo:

- `training_precision_score`
- `training_recall_score`
- `training_f1_score`
- `training_accuracy_score`
- `training_log_loss`
- `training_roc_auc`
- `training_score`
- `LogisticRegression_score_X_test`

Cada archivo contiene el valor de la métrica y su timestamp.


### 3. Parámetros del modelo (`params/`)

Todos los hiperparámetros usados en la regresión logística:

- `C`, `penalty`, `solver`, `max_iter`, `random_state`, `class_weight`, `tol`, etc.
- También se registraron opciones avanzadas como `warm_start`, `multi_class`, `intercept_scaling`, `n_jobs`.

Esto permite reproducir exactamente la configuración del modelo.


### 4. Etiquetas (`tags/`)

Incluyen metadatos adicionales:

- `mlflow.user`: usuario que ejecutó el experimento.
- `mlflow.source.name`: nombre del script o notebook.
- `mlflow.source.type`: tipo de fuente (por ejemplo `notebook`).
- `mlflow.runName`: nombre de la ejecución.
- `estimator_name` y `estimator_class`: identifican el tipo de modelo usado.
- `mlflow.log-model.history`: historial de logueo del modelo.


### 5. Artefactos (`artifacts/`)

Estos son los archivos generados y guardados:

#### Visualizaciones

- `training_confusion_matrix.png`: matriz de confusión del entrenamiento.
- `estimator.html`: una visualización generada automáticamente del modelo en html, útil si utilizamos pipelines de scikit-learn con múltiples pasos. 

#### Modelo guardado (`model/`)

Estructura típica de MLflow para modelos:

- `model.pkl`: el modelo serializado (en formato `pickle`).
- `MLmodel`: archivo de configuración que describe cómo cargar e interpretar el modelo.
- `conda.yaml`, `requirements.txt`, `python_env.yaml`: descripción del entorno necesario para reproducir o servir el modelo.

Observemos en qué consiste el MLmodel:

```yaml
artifact_path: model
flavors:
  python_function:
    env:
      conda: conda.yaml
      virtualenv: python_env.yaml
    loader_module: mlflow.sklearn
    model_path: model.pkl
    predict_fn: predict
    python_version: 3.10.13
  sklearn:
    code: null
    pickled_model: model.pkl
    serialization_format: cloudpickle
    sklearn_version: 1.6.1
is_signature_from_type_hint: false
mlflow_version: 2.21.3
model_size_bytes: 836
model_uuid: 1458474b44934db9868e673d2f774503
prompts: null
run_id: d34a23d767d74cdfa143efd3660a0010
signature:
  inputs: '[{"type": "tensor", "tensor-spec": {"dtype": "float64", "shape": [-1, 4]}}]'
  outputs: '[{"type": "tensor", "tensor-spec": {"dtype": "int64", "shape": [-1]}}]'
  params: null
type_hint_from_example: false
utc_time_created: '2025-04-18 22:33:15.610355'
```

*NOTA*: se han omitido algunos ficheros no relevantes.

### ¿Qué cubrimos con estos datos?

- **Reproducir el entrenamiento**: tienes los parámetros y métricas.
- **Servir el modelo**: puedes cargar el `model.pkl` con MLflow o directamente si replicas el entorno.
- **Auditoría y trazabilidad**: puedes ver cuándo, cómo y con qué se entrenó.
- **Evaluación y comparación**: comparar ejecuciones usando métricas registradas.


In [9]:
# !ls -l mlruns # en caso de tener linux/mac, 
# !dir mlruns # en caso de tener windows

Si volvemos a ejecutar otro modelo, se consideraría que hemos realizado un nuevo **experimento**. Y toda su información se almacenaría en la misma ruta, pero separando ambos en diferentes carpetas (0 y 1).


NOTA: la carpeta .trash es necesaria en la raíz con las carpetas de experimentos (1-n) o tendremos un error.

In [7]:
with mlflow.start_run():
    clf = LogisticRegression(random_state=42, fit_intercept=False, penalty='l2') # modificamos algunos de los parámetros
    clf.fit(X_train, y_train)
    clf.score(X_test, y_test)



La carpeta `mlruns` contendrá una carpeta con un número secuencial que identifica el experimento (*6d34a3cc385a4d33b4f40d886510381b*). La estructura de la carpeta aparecerá de la siguiente manera:


```{figure} img/mlflow/runs_root.png
---
name: segundo experimento
width: 80%
---
Estructura segundo experimento.
```


## Módulos de MLflow 

Los módulos de MLflow son componentes de software que proporcionan las características principales que ayudan en las diferentes fases del ciclo de vida de ML:

- **MLflow Tracking**: Proporciona un mecanismo y una interfaz de usuario para manejar métricas y artefactos generados por ejecuciones de ML (entrenamiento e inferencia)
- **MLflow Projects**: Un formato de paquete para estandarizar proyectos de ML
- **MLflow Models**: Un mecanismo que se implementa en diferentes tipos de entornos, tanto locales como en la nube
- **MLflow Model Registry**: Un módulo que maneja la gestión de modelos en MLflow y su ciclo de vida, incluyendo el estado
    
Exploraremos estos módulos en las siguientes secciones.


### mlflow.autolog()

El código siguiente muestra las operaciones(*) que estamos usando por el hecho de aplicar autolog.

```python
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Importante si antes ejecutaste autolog en el notebook
mlflow.sklearn.autolog(disable=True)

mlflow.set_experiment("my_experiment_1")

with mlflow.start_run():
    clf = LogisticRegression(random_state=99)
    clf.fit(X_train, y_train)

    # Métrica equivalente a clf.score(...)
    y_pred = clf.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred)

    # Params del modelo
    mlflow.log_params(clf.get_params())

    # Métricas
    mlflow.log_metric("test_accuracy", test_accuracy)

    # Modelo como artefacto
    signature = infer_signature(X_train, clf.predict(X_train))
    mlflow.sklearn.log_model(
        sk_model=clf,
        artifact_path="model",
        signature=signature,
        input_example=X_train[:5]
    )
```

In [7]:
## ¿Dónde está el modelo?

print("Tracking URI:", mlflow.get_tracking_uri())

exp = mlflow.get_experiment_by_name("my_experiment_1")
print("Experiment ID:", exp.experiment_id)
print("Artifact location:", exp.artifact_location)


Tracking URI: file:///home/johndoe/Documents/software/mlops-course/course-core/jupyterbook/mlruns
Experiment ID: 686985879940142405
Artifact location: file:///home/johndoe/Documents/software/mlops-course/mlbook/mlruns/686985879940142405
